# 02 Train, evaluate, export

Kaggle GPU notebook. Estimated runtime: 5-20 min. Attach notebook 01 outputs and this repository as Kaggle inputs.

In [ ]:
from pathlib import Path
import random,json,subprocess,shutil
import numpy as np
SEED=42
TOP_K=200
MODEL_NAME="facebook/esm2_t6_8M_UR50D"
DATA_DIR = "/kaggle/input/competitions/cafa-5-protein-function-prediction"
FASTA=f"{DATA_DIR}/Train/train_sequences.fasta"
TERMS=f"{DATA_DIR}/Train/train_terms.tsv"
OBO=f"{DATA_DIR}/Train/go-basic.obo"
TAXONOMY=f"{DATA_DIR}/Train/train_taxonomy.tsv"
IA_FILE=f"{DATA_DIR}/IA.txt"
WORK=Path("/kaggle/working/protfunc")
random.seed(SEED); np.random.seed(SEED)
missing=[p for p in [FASTA,TERMS,OBO,TAXONOMY] if not Path(p).is_file()]
if missing: raise FileNotFoundError("Missing CAFA inputs: "+", ".join(missing))
if not Path(IA_FILE).is_file(): print("Optional IA.txt unavailable; S-min is outside this reduced scope.")
WORK.mkdir(parents=True,exist_ok=True)
print("All required CAFA input files exist")
import glob
source_hits=glob.glob("/kaggle/input/**/src/protfunc/models.py",recursive=True)
if not source_hits: raise FileNotFoundError("Could not find src/protfunc/models.py under /kaggle/input")
SOURCE=Path(source_hits[0]).parents[2]
split_hits=sorted(glob.glob("/kaggle/input/**/splits.json",recursive=True)+glob.glob("/kaggle/working/**/splits.json",recursive=True))
if not split_hits: raise FileNotFoundError("Could not find splits.json under /kaggle/input or /kaggle/working")
PREP=Path(split_hits[0]).parent
assert (PREP/"embeddings").is_dir(),f"Prepared embeddings directory missing under {PREP}"

Verify inputs

In [ ]:
subprocess.run(["python","-m","pip","install","-q","-r",str(SOURCE/"requirements-train.txt")],check=True)
assert all(Path(p).is_file() for p in [FASTA,TERMS,OBO,TAXONOMY]),"Required CAFA input missing"
assert (PREP/"splits.json").is_file() and list((PREP/"embeddings").glob("emb_*.npy")),"Prepared split or embedding shards missing"
print("All source, CAFA, and prepared files exist")

Train and export

In [ ]:
# SOURCE and PREP were found by glob in the setup cell.
import sys,runpy
sys.path.insert(0,str(SOURCE/"src")); sys.path.insert(0,str(SOURCE/"notebooks"))
sys.argv=["train_gpu.py","--data-dir",DATA_DIR,"--prep-dir",str(PREP),"--output-dir","/kaggle/working"]
runpy.run_path(str(SOURCE/"notebooks/train_gpu.py"),run_name="__main__")

## Optional DIAMOND top-hit baseline
This uses training sequences only as the database, transfers top-hit annotations weighted by bitscore, and appends measured metrics. Skip this section if time is short.

In [ ]:
# Optional: run only if time permits. Database is built from the training split only.
import subprocess,sys,runpy
subprocess.run(["apt-get","update","-qq"],check=True)
subprocess.run(["apt-get","install","-y","-qq","diamond-aligner"],check=True)
sys.argv=["run_diamond.py",DATA_DIR,str(PREP),"/kaggle/working"]
runpy.run_path(str(SOURCE/"notebooks/run_diamond.py"),run_name="__main__")